In [0]:
import os
import glob
import pandas as pd
from tqdm import tqdm

# 1. Procura automaticamente os arquivos Danos_Informados_*.csv dentro do Volume fontes_tcc
CAMINHO_VOLUME = "/Volumes/workspace/default/fontes_tcc"
padrao_busca = os.path.join(CAMINHO_VOLUME, "**", "Danos_Informados*.csv")
arquivos_danos = sorted(glob.glob(padrao_busca, recursive=True))

print(f"Total de arquivos anuais encontrados no Volume: {len(arquivos_danos)}")
for arq in arquivos_danos:
    print(f" -> {os.path.basename(arq)}")

# 2. Inspecionando as primeiras linhas do arquivo de 2021 para comprovar o skiprows=4
if len(arquivos_danos) > 0:
    arq_teste = arquivos_danos[0]
    print(f"\n--- INSPEÇÃO DO CABEÇALHO BRUTO ({os.path.basename(arq_teste)}) ---")
    with open(arq_teste, "r", encoding="latin1") as f:
        for i in range(7):
            print(f"Linha {i}: {f.readline().strip()}")
            
    # 3. Lendo e empilhando os 5 arquivos anuais (skiprows=4 pula as 4 linhas iniciais do S2iD)
    print("\n" + "=" * 65)
    print(" EMPILHANDO ARQUIVOS ANUAIS DE DANOS INFORMADOS (S2iD 2021-2025)")
    print("=" * 65)
    
    lista_dfs_danos = []
    for caminho_arq in tqdm(arquivos_danos, desc="Processando lotes anuais"):
        df_ano = pd.read_csv(caminho_arq, sep=";", encoding="latin1", skiprows=4, on_bad_lines="skip")
        df_ano["Arquivo_Origem"] = os.path.basename(caminho_arq)
        lista_dfs_danos.append(df_ano)
        print(f"   * {os.path.basename(caminho_arq)}: {len(df_ano)} linhas")
        
    df_s2id_bruto = pd.concat(lista_dfs_danos, ignore_index=True)
    
    print("-" * 65)
    print(f"[SUCESSO] Total de colunas identificadas: {len(df_s2id_bruto.columns)}")
    print(f"[SUCESSO] Primeiras 10 colunas: {df_s2id_bruto.columns[:10].tolist()}")
    print(f"[VERIFICAÇÃO] Coluna 'DH_Desalojados' presente? {'DH_Desalojados' in df_s2id_bruto.columns}")
    print(f"[VERIFICAÇÃO] Coluna 'Código IBGE' presente? {'Código IBGE' in df_s2id_bruto.columns}")
    print(f"[SUCESSO] Total de registros empilhados (2021-2025): {len(df_s2id_bruto)} linhas")
else:
    print("[AVISO] Nenhum arquivo encontrado. Verifique se o upload foi concluído.")
    

Total de arquivos anuais encontrados no Volume: 5
 -> Danos_Informados_2021.csv
 -> Danos_Informados_2022.csv
 -> Danos_Informados_2023.csv
 -> Danos_Informados_2024.csv
 -> Danos_Informados_2025.csv

--- INSPEÇÃO DO CABEÇALHO BRUTO (Danos_Informados_2021.csv) ---
Linha 0: Ministério da Integração e do Desenvolvimento Regional
Linha 1: Secretaria Nacional de Proteção e Defesa Civil
Linha 2: Relatório Gerencial - Danos Informados
Linha 3: Data Inicial: 01/01/2021 Data Final: 31/12/2021 UF: AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,MG,MS,MT,PA,PB,PE,PI,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
Linha 4: UF;Município;Registro;Protocolo;COBRADE;Status;População;DH_Mortos;DH_Feridos;DH_Enfermos;DH_Desabrigados;DH_Desalojados;DH_Desaparecidos;DH_Outros Afetados;DM_Unidades Habitacionais Danificadas;DM_Unidades Habitacionais Destruídas;DM_Unidades Habitacionais Valor;DM_Instalações públicas de saúde Danificadas;DM_Instalações públicas de saúde Destruídas;DM_Instalações públicas de saúde Valor;DM_Instalações públicas 

Processando lotes anuais: 100%|██████████| 5/5 [00:00<00:00, 30.72it/s]

   * Danos_Informados_2021.csv: 12002 linhas
   * Danos_Informados_2022.csv: 6831 linhas
   * Danos_Informados_2023.csv: 6414 linhas
   * Danos_Informados_2024.csv: 6032 linhas
   * Danos_Informados_2025.csv: 6368 linhas
-----------------------------------------------------------------
[SUCESSO] Total de colunas identificadas: 54
[SUCESSO] Primeiras 10 colunas: ['UF', 'Município', 'Registro', 'Protocolo', 'COBRADE', 'Status', 'População', 'DH_Mortos', 'DH_Feridos', 'DH_Enfermos']
[VERIFICAÇÃO] Coluna 'DH_Desalojados' presente? True
[VERIFICAÇÃO] Coluna 'Código IBGE' presente? False
[SUCESSO] Total de registros empilhados (2021-2025): 37647 linhas


In [0]:
# Célula 2: Salvar a base empilhada de Danos Informados no Volume
caminho_danos_consolidado = "/Volumes/workspace/default/fontes_tcc/S2iD_Danos_Informados_Bruto_2021_2025.csv"
df_s2id_bruto.to_csv(caminho_danos_consolidado, index=False, sep=";", encoding="utf-8-sig")
print(f"[SUCESSO] Base consolidada de Danos salva em: {caminho_danos_consolidado}")

[SUCESSO] Base consolidada de Danos salva em: /Volumes/workspace/default/fontes_tcc/S2iD_Danos_Informados_Bruto_2021_2025.csv
